# Can an auditory figure survive being sheared in time?
### A playground built to be attacked

**Temporal coherence theory** says components that start together bind into one perceptual object.
The standard test is the *stochastic figure-ground* stimulus: a dense cloud of short tone pips, inside
which a small set of channels repeatedly sounds together. Listeners hear that set detach from the cloud
as a "figure" even though no single instant of the stimulus is distinctive.

Published work shears the figure in **frequency**. Nobody has sheared it in **time** — which is the
harder case, because onset asynchrony is precisely what the theory says should destroy binding.

> **The question.** How far apart can a figure's components be pulled in onset time before it stops
> being one thing? And does *detecting that a pattern recurs* fail before *detecting that anything
> groups at all* does?

---

### How to read this notebook

You are the skeptic. Each section states an objection **in your voice**, then answers it with a
measurement you can re-run, a plot you can inspect, or a sound you can listen to. Nothing here asks
you to take a construction argument on faith.

Two things make this different from a demo:

1. **You can listen to the confounds directly.** Section 3 plays you the two intervals reduced to
   *only* their long-term spectrum; section 4 plays them reduced to *only* their amplitude envelope.
   If you cannot tell those apart, that cue cannot be what a listener uses.
2. **You can break it.** Section 6 plants a confound on purpose and watches the verification battery
   catch it. A test that cannot fail is not evidence.

**Headphones required.** Set a comfortable level on the first sound and leave it there.

*Runtime: about 5 minutes. Every cell is independent of your answers, so you can skip any of them.*

In [ ]:
#@title Setup — clone the repository and import (run me first)
import os, sys, subprocess, textwrap

REPO_URL  = "https://github.com/MeysamAmirsardari/SeqSFG_task.git"
REPO_DIR  = "SeqSFG_task"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)

# Work whether we're in Colab (repo just cloned) or inside a local checkout.
ROOT = os.path.abspath(REPO_DIR) if os.path.isdir(REPO_DIR) else os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown

from seqsfg import config, stimulus, measure, verify, analysis, plots
from seqsfg import pool as poolmod
from seqsfg.stimulus import make_trial, render_interval, render_trial, FIGURE, BACKGROUND

cfg = config.DEFAULT
D   = config.validate(cfg)
SR  = cfg.sample_rate

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False, "font.size": 9})

print(config.describe(cfg))
print("\nrepository:", ROOT)

---

## 0. The stimulus, in one paragraph

Both intervals of a trial contain **the same 744 tones**, 31 in every channel of a 24-channel pool,
over 2.25 s. In the *target* interval, six figure **elements** of seven components each arrive at
3–5 Hz, always on the same seven channels. What the *other* interval holds is the manipulation:

| ladder | the other interval holds | falling performance means |
|---|---|---|
| `rising` | six elements too, each on **seven freshly drawn channels** | the listener stopped detecting **recurrence** |
| `ungrouped` | **no elements at all** | the listener stopped detecting **presence** |

The independent variable is `step`: the onset delay between successive components of one element.
At `step = 0` an element is a chord; at `step = 28 ms` it is a staircase spread over 198 ms.

**The one design decision everything rests on:** figure tones are *not added* to the background, they
are **scheduled out of each channel's fixed budget**. So every channel carries the same number of
tones in every interval, and the long-term spectrum is flat by construction. That is what makes the
comparison honest — and it is the first thing you should try to break.

In [ ]:
#@title Helpers — playback, and two ways to destroy information on purpose
from scipy.signal import hilbert
from scipy.ndimage import uniform_filter1d

def play(x, label="", normalize=False):
    """Play a mono float array. normalize=False keeps LEVEL COMPARISONS honest."""
    if label:
        display(Markdown(f"**{label}**"))
    display(Audio(np.asarray(x, dtype=float), rate=SR, normalize=normalize))

def broadband_envelope(x, smooth_ms=4.0):
    """The amplitude envelope a listener's broadband level meter would see."""
    a = np.abs(hilbert(np.asarray(x, dtype=float)))
    return uniform_filter1d(a, size=max(int(SR * smooth_ms / 1000.0), 1))

def envelope_only(x, seed=0):
    """Keep ONLY the broadband envelope: impose it on white noise, discard all spectral detail.
    If two intervals sound the same here, the envelope cannot be what tells them apart."""
    rng = np.random.default_rng(seed)
    n = rng.normal(size=len(x))
    n /= np.max(np.abs(n)) + 1e-12
    return broadband_envelope(x) * n * 3.0

def spectrum_only(x, seed=0, dur_s=None):
    """Keep ONLY the long-term spectrum: re-emit each channel as a steady tone whose amplitude
    matches that channel's long-term power, with all timing discarded."""
    rng = np.random.default_rng(seed)
    freqs = D.channel_freqs_hz
    env = measure.channel_envelopes(x, SR, freqs, win_ms=40.0, hop_ms=1.0)
    amp = np.sqrt(np.mean(env ** 2, axis=1))
    n = len(x) if dur_s is None else int(dur_s * SR)
    t = np.arange(n) / SR
    out = np.zeros(n)
    for f, a in zip(freqs, amp):
        out += a * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi))
    r = int(0.02 * SR)
    ramp = 0.5 * (1 - np.cos(np.pi * np.arange(r) / r))
    out[:r] *= ramp; out[-r:] *= ramp[::-1]
    return out

def figure_subset(iv, keep=FIGURE):
    """A copy of an interval containing only its figure tones (or only its background)."""
    m = iv.kind == keep
    out = iv.copy()
    for a in ("onset", "channel", "phase", "kind", "element", "component"):
        setattr(out, a, getattr(iv, a)[m])
    return out

def raster(ax, iv, t0=0.0, t1=None, lw=2.4, mark_figure=True, boxes=False):
    plots._raster(ax, cfg, D, iv, t0, cfg.interval_dur_ms / 1000.0 if t1 is None else t1,
                  lw=lw, boxes=boxes, mark_figure=mark_figure)
    ax.set_xlabel("time (s)"); ax.set_ylabel("semitones re 1 kHz")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

print("helpers ready")

---

## 1. Before any argument: listen

A skeptic should hear the thing before reading claims about it. Below, one trial at `step = 0`
(the easiest case, where an element is a chord), taken apart:

1. **The figure alone** — the seven channels that recur, with the cloud removed. This is what you are
   being asked to hear.
2. **The background alone** — the same interval with the figure removed.
3. **The mixture** — the actual stimulus. The figure is now at the same level as everything else,
   and it is *not* louder, *not* spectrally marked, and *not* added on top.

If the figure vanishes for you in the mixture, that is the honest starting point of this experiment,
and section 2 will let you find out whether it is really gone or just hard.

In [ ]:
#@title The figure, the ground, and the mixture — one trial at step 0
trial = make_trial(cfg, seed=20260902, step_ms=0.0, variant="rising", d=D)
A     = trial.recurring

x_fig  = render_interval(cfg, figure_subset(A, FIGURE), D)
x_bg   = render_interval(cfg, figure_subset(A, BACKGROUND), D)
x_mix  = render_interval(cfg, A, D)

play(x_fig, "1. The figure alone — six chords on the same seven channels (this is the target)")
play(x_bg,  "2. The background alone — same cloud, figure removed")
play(x_mix, "3. The mixture — the actual stimulus. Same total tones as any other interval.")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, iv, title in ((axes[0], figure_subset(A, FIGURE), "figure alone"),
                      (axes[1], figure_subset(A, BACKGROUND), "background alone"),
                      (axes[2], A, "the mixture")):
    raster(ax, iv, boxes=(title == "the mixture"))
    ax.set_title(title)
axes[1].set_ylabel(""); axes[2].set_ylabel("")
plt.tight_layout(); plt.show()

print(f"figure tones: {int(np.sum(A.kind == FIGURE))}   background tones: {int(np.sum(A.kind == BACKGROUND))}"
      f"   total: {A.n_tones}")
print("tones per channel:", np.bincount(A.channel, minlength=D.n_channels))

### The figure emerges if you lower the cloud

Here is the same trial with the background attenuated by 24, 18, 12, 6 and 0 dB. Listen down the list.
The pattern you learn to hear at −24 dB is *still physically present* at 0 dB — the last sound is
exactly the stimulus from section 1.

This is not a trick to make the task look easy. It is a way to teach your ear what to listen for, and
it is why the real experiment has a practice block with feedback before anything is measured.

In [ ]:
#@title Figure-to-ground sweep — the same trial, cloud brought up in 6 dB steps
for db in (-24, -18, -12, -6, 0):
    y = x_fig + x_bg * (10 ** (db / 20.0))
    play(y, f"background at {db:+d} dB relative to the stimulus" + ("   <-- the real stimulus" if db == 0 else ""))

---

## 2. Take the test yourself

Two intervals per trial. **One of them keeps bringing a group back at the same pitches.** The other
brings a group back at *new* pitches every time. Which was it, 1 or 2?

Run the cell, type `1` or `2`, press Enter. Type `q` to stop early. You get feedback after each trial,
exactly as a real participant does in the practice block.

Start with `STEP_MS = 0`. Then re-run with `STEP_MS = 15` and `28` and watch what happens to your own
accuracy — that is the experiment, performed on you, with n = 1.

In [ ]:
#@title Mini 2IFC block — you are the listener   { run: "auto" }
STEP_MS  = 0      #@param [0, 5, 10, 15, 20, 28]
N_TRIALS = 6      #@param {type:"slider", min:2, max:12, step:1}
VARIANT  = "rising"   #@param ["rising", "ungrouped"]

def ask(prompt):
    try:
        return input(prompt).strip().lower()
    except (EOFError, OSError):
        return "q"

def self_test(step_ms=0.0, n_trials=6, variant="rising", seed=7):
    rng = np.random.default_rng(seed)
    score = 0
    done  = 0
    for i in range(n_trials):
        tgt = int(rng.integers(1, 3))
        tr  = make_trial(cfg, seed=int(rng.integers(1, 2**31 - 1)), step_ms=float(step_ms),
                         variant=variant, d=D)
        x   = render_trial(cfg, tr, tgt, D)
        display(Markdown(f"### Trial {i+1} of {n_trials}"))
        play(x)
        ans = ask("Which interval kept coming back at the same pitches?  [1/2, q to quit]  ")
        if ans == "q" or ans not in ("1", "2"):
            if ans != "q":
                print("(no input available — stopping)")
            break
        done += 1
        ok = int(ans) == tgt
        score += ok
        print("correct" if ok else f"wrong — it was interval {tgt}")
    if done:
        print(f"\nYou scored {score}/{done} at step = {step_ms:g} ms on the '{variant}' ladder.")
        print("Chance is 50%. With this few trials, treat it as a demonstration, not a measurement.")
    return score, done

self_test(STEP_MS, N_TRIALS, VARIANT)

---

# The objections

From here on, each section is an objection stated in your voice, and an answer you can re-run.

---

## 3. *"You've just made the figure's channels louder. I'd be hearing energy, not binding."*

This is the objection that kills most versions of this stimulus, and it is why the per-channel budget
exists. Two ways to check it.

**First, measure it.** Take the long-term spectrum of both intervals of a trial, channel by channel,
and take the strategy a cheater would use: *find how far the most prominent channels stand above the
rest, and pick the interval with the taller peaks.*

**Second — and this is the part you can judge with your ears — listen to it.** The cell below
re-synthesises each interval as a steady chord whose per-channel amplitudes match that interval's
long-term spectrum exactly, with **all timing thrown away**. If a listener really were reading the
spectrum, these two sounds would be as different as the intervals are. Play them.

In [ ]:
#@title The spectrum of both intervals — measured, then played
tr = make_trial(cfg, seed=4242, step_ms=0.0, variant="rising", d=D)
xa = render_interval(cfg, tr.recurring, D)   # A: the figure recurs on the same channels
xb = render_interval(cfg, tr.other,     D)   # B: the figure lands on new channels every element

freqs = D.channel_freqs_hz
spec_a = measure.channel_power_db(measure.channel_envelopes(xa, SR, freqs))
spec_b = measure.channel_power_db(measure.channel_envelopes(xb, SR, freqs))
S = tr.recurring.figure_set          # which channels recur (the listener is NOT told this)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].semilogx(freqs, spec_a - spec_a.mean(), "o-", label="A: figure recurs here", ms=4)
ax[0].semilogx(freqs, spec_b - spec_b.mean(), "s--", label="B: figure moves each time", ms=4)
ax[0].semilogx(freqs[S], (spec_a - spec_a.mean())[S], "rv", ms=9,
               label="A's recurring channels")
ax[0].set(xlabel="channel frequency (Hz)", ylabel="long-term level re mean (dB)",
          title="the recurring channels leave no mark")
ax[0].set_xticks([250, 500, 1000, 2000, 4000])
ax[0].get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax[0].legend(fontsize=7.5); ax[0].grid(alpha=0.25)

diff = spec_a - spec_b
ax[1].bar(range(len(diff)), diff, color="0.4")
ax[1].set(xlabel="channel index", ylabel="A − B (dB)",
          title=f"per-channel difference: max |A−B| = {np.abs(diff).max():.3f} dB")
plt.tight_layout(); plt.show()

pk = lambda v: np.mean(np.sort(v)[::-1][:cfg.n_components]) - np.median(v)
print(f"the cheater's statistic (top-{cfg.n_components} channels minus the median):")
print(f"   interval A: {pk(spec_a):+.4f} dB      interval B: {pk(spec_b):+.4f} dB")
print(f"   difference: {pk(spec_a) - pk(spec_b):+.4f} dB  <- pure measurement noise\n")

play(spectrum_only(xa, dur_s=2.0), "Interval A, reduced to its long-term spectrum alone")
play(spectrum_only(xb, dur_s=2.0), "Interval B, reduced to its long-term spectrum alone")
display(Markdown("If you can hear which of those two contained the recurring figure, the design is "
                 "broken and I want to know. They are the same chord."))

---

## 4. *"Then it's a level bump. Seven tones starting together is a thump, and I'd hear the thump."*

A fair and much harder objection — and it is the trap that broke an earlier version of this code.

The answer is not that there is no thump. **There is.** Both intervals contain elements, so both
produce an element-locked level transient. The question is whether the transient *differs* between
them, because only a difference is a cue.

Below: the broadband envelope averaged over element onsets, for both intervals, with a 95% interval.
Then the same destructive test as before — each interval reduced to **only its amplitude envelope**,
imposed on white noise, with all spectral detail discarded. Listen for which one contains the
recurring figure.

In [ ]:
#@title The element-locked envelope — measured, then played
def locked(x, iv, span_ms, pre_ms=60.0, post_ms=120.0, frame_ms=2.0):
    env  = measure.frame_rms(x, SR, frame_ms)
    return measure.element_locked_envelope(env, frame_ms, iv.element_onsets * cfg.grid_ms,
                                           span_ms, pre_ms, post_ms)

span = (cfg.n_components - 1) * 0.0 + cfg.tone_dur_ms
segs_a, segs_b = [], []
for k in range(24):                       # 24 fresh trials so the average means something
    t_ = make_trial(cfg, seed=9000 + k, step_ms=0.0, variant="rising", d=D)
    segs_a.append(locked(render_interval(cfg, t_.recurring, D), t_.recurring, span))
    segs_b.append(locked(render_interval(cfg, t_.other,     D), t_.other,     span))
Aseg, Bseg = np.array(segs_a), np.array(segs_b)
t_ms = np.arange(Aseg.shape[1]) * 2.0 - 60.0

fig, ax = plt.subplots(figsize=(8.5, 3.8))
for Y, c, lbl in ((Aseg, "#1f77b4", "A: figure recurs"), (Bseg, "#ff7f0e", "B: figure moves")):
    mu, se = Y.mean(0), Y.std(0, ddof=1) / np.sqrt(len(Y))
    ax.plot(t_ms, mu, color=c, lw=1.6, label=lbl)
    ax.fill_between(t_ms, mu - 1.96 * se, mu + 1.96 * se, color=c, alpha=0.3, lw=0)
ax.axvspan(0, span, color="0.85", alpha=0.6, zorder=0)
ax.axvline(0, color="0.5", ls=":", lw=0.9)
ax.set(xlabel="time re element onset (ms)", ylabel="broadband level re interval RMS (dB)",
       title="Both intervals thump. The thump is the same size.")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"peak of the transient:  A {Aseg.mean(0).max():+.2f} dB      B {Bseg.mean(0).max():+.2f} dB")
print(f"largest difference anywhere in the window: {np.abs(Aseg.mean(0) - Bseg.mean(0)).max():.2f} dB\n")

play(envelope_only(xa), "Interval A, reduced to its amplitude envelope alone")
play(envelope_only(xb), "Interval B, reduced to its amplitude envelope alone")
display(Markdown("Two noise bursts with identical statistics. The envelope is not the cue."))

---

## 5. *"Your ideal observers are toys. Of course a hand-picked statistic finds nothing."*

Right — so don't hand-pick. The battery extracts **62 scalar features** from each interval (spectral
peakedness, occupancy, modulation depth and its spectrum, per-channel inter-onset statistics,
periodicity at element-rate lags, level spread, …) and asks a single corrected question:

> Does the **largest** effect across all 62 features exceed what **relabelling** produces?

Exchanging the two intervals of a trial *is* the null hypothesis "these differ in nothing an observer
can measure". Flipping that label on a random subset of trials gives the exact null distribution of
the whole audit, corrected for having looked at every feature at once.

The cell builds fresh trials on your machine — a different random draw from the one in the repository —
and runs it live.

In [ ]:
#@title Live: all 62 features against the relabelling null (~60 s)
N_PER_CELL = 16   #@param {type:"slider", min:8, max:32, step:4}

res = verify.run_battery(cfg, n_trials=N_PER_CELL, seed=int(np.random.default_rng().integers(1e6)),
                         conditions=[("rising", s) for s in cfg.steps_ms], verbose=False)
pm = res["permutation"]

fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1.4, 1]})
d_ = np.array([r["dprime"] for r in res["audit"]]); order = np.argsort(d_)
ax[0].errorbar(d_[order], np.arange(len(d_)),
               xerr=[d_[order] - np.array([res["audit"][i]["ci"][0] for i in order]),
                     np.array([res["audit"][i]["ci"][1] for i in order]) - d_[order]],
               fmt="o", ms=3, color="0.25", capsize=1.5, lw=0.8)
ax[0].axvspan(-pm["null_max_p95"], pm["null_max_p95"], color="0.75", alpha=0.3, lw=0,
              label=f"95% band for the largest of {pm['n_features']}")
ax[0].axvline(0, color="0.3", lw=1)
ax[0].set(xlabel="fixed-rule d'", ylabel="feature (sorted)", yticks=[],
          title=f"every feature, {pm['n_trials']} trials")
ax[0].legend(fontsize=8, loc="lower right")

ax[1].hist(pm["null_max"], bins=50, color="0.6", label="largest |d'| when the\nintervals are exchanged")
ax[1].axvline(pm["observed_max_dprime"], color="#d62728", lw=2.2,
              label=f"observed = {pm['observed_max_dprime']:.2f}\n({pm['worst_feature']})")
ax[1].set(xlabel="largest |d'| over all features", ylabel="count",
          title=f"permutation test:  p = {pm['p_value']:.3f}")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"largest effect any of {pm['n_features']} features produces : {pm['observed_max_dprime']:.3f}")
print(f"typical largest effect under relabelling            : {pm['null_max_median']:.3f}")
print(f"p (any feature separates the intervals)             : {pm['p_value']:.3f}")
print("\nper-observer, pooled over the ladder (leave-one-out within each condition):")
for nm, r in res["pooled_main"].items():
    tag = "  <- oracle, told when the elements are" if nm.startswith("oracle") else \
          f"  Holm p = {r.get('p_holm', float('nan')):.3f}"
    print(f"   {nm:30s} d' = {r['dprime']:+.3f}  [{r['ci'][0]:+.2f}, {r['ci'][1]:+.2f}]{tag}")

---

## 6. *"A test that never fires isn't evidence. Show me it can fail."*

The right demand, and the easiest to satisfy.

Below we **deliberately break the design** in the exact way the per-channel budget was invented to
prevent: instead of *substituting* the figure's tones out of each channel's budget, we **add** them on
top. The figure's channels now carry seven extra tones' worth of energy. This is the naive version of
the stimulus, and it is what most implementations do.

Nothing else changes. Watch the spectrum observer — at chance in section 5 — light up.

In [ ]:
#@title Plant a confound: ADD the figure instead of substituting it
def render_added(trial):
    """The broken design: figure energy on top of a full background."""
    A = trial.recurring
    return render_interval(cfg, A, D) + render_interval(cfg, figure_subset(A, FIGURE), D)

def peakedness_of(x):
    v = measure.channel_power_db(measure.channel_envelopes(x, SR, D.channel_freqs_hz))
    return np.mean(np.sort(v)[::-1][:cfg.n_components]) - np.median(v)

good_a, good_b, bad_a, bad_b = [], [], [], []
for k in range(24):
    t_ = make_trial(cfg, seed=31000 + k, step_ms=0.0, variant="rising", d=D)
    good_a.append(peakedness_of(render_interval(cfg, t_.recurring, D)))
    good_b.append(peakedness_of(render_interval(cfg, t_.other, D)))
    bad_a.append(peakedness_of(render_added(t_)))
    bad_b.append(peakedness_of(render_interval(cfg, t_.other, D)))

ob_good = verify.observer_from_stat(np.array(good_a), np.array(good_b))
ob_bad  = verify.observer_from_stat(np.array(bad_a),  np.array(bad_b))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
for a, (ga, gb, ob, title) in zip(ax, [(good_a, good_b, ob_good, "as designed: figure SUBSTITUTED"),
                                       (bad_a,  bad_b,  ob_bad,  "broken: figure ADDED on top")]):
    bins = np.linspace(min(min(ga), min(gb)), max(max(ga), max(gb)), 22)
    a.hist(ga, bins=bins, alpha=0.65, label="A (figure recurs)", color="#1f77b4")
    a.hist(gb, bins=bins, alpha=0.65, label="B", color="#ff7f0e")
    a.set(xlabel="spectral peakedness (dB)", title=f"{title}\nspectrum observer d' = {ob['dprime']:+.2f}"
                                                   f"  (p = {ob['p']:.4f})")
    a.legend(fontsize=8)
ax[0].set_ylabel("count")
plt.tight_layout(); plt.show()

print(f"as designed : d' = {ob_good['dprime']:+.2f}   p = {ob_good['p']:.4f}   <- at chance")
print(f"broken      : d' = {ob_bad['dprime']:+.2f}   p = {ob_bad['p']:.4f}   <- caught\n")

t_demo = make_trial(cfg, seed=31000, step_ms=0.0, variant="rising", d=D)
play(render_interval(cfg, t_demo.recurring, D), "As designed — the figure is in there, at the same level as everything else")
play(render_added(t_demo), "Broken — the same figure ADDED on top. Hear how it pops out?")

---

## 7. *"Fine. Now tell me what you could NOT fix."*

Two things. Both are stated with numbers rather than hedges.

### 7a. The `ungrouped` ladder has a non-binding route at its smallest steps — necessarily

A figure that is present in one interval and absent in the other **is** a level event. Matching the
envelope would mean giving the foil synchronous onsets, which is giving it a group. So the trichotomy
is real: a foil is either grouped (and then it is the `rising` foil) or ungrouped (and then its
envelope differs). This cannot be designed away, only measured and bounded.

Listen to it, then look at where it lives.

In [ ]:
#@title The one cue that cannot be removed — hear it, then see where it lives
t0 = make_trial(cfg, seed=555, step_ms=0.0,  variant="ungrouped", d=D)
t28 = make_trial(cfg, seed=555, step_ms=28.0, variant="ungrouped", d=D)

display(Markdown("**At step 0** the target is a chord and the foil has no group at all — "
                 "envelope-only versions, which should be indistinguishable if the cue were absent:"))
play(envelope_only(render_interval(cfg, t0.recurring, D)), "step 0, target (envelope only)")
play(envelope_only(render_interval(cfg, t0.other,     D)), "step 0, foil (envelope only)")
display(Markdown("You can probably hear the difference. That is the cue, and it is why the "
                 "`ungrouped` curve cannot be read at face value at step 0.\n\n"
                 "**At step 28 ms** the same seven components are spread over 198 ms and no longer "
                 "make a transient:"))
play(envelope_only(render_interval(cfg, t28.recurring, D)), "step 28, target (envelope only)")
play(envelope_only(render_interval(cfg, t28.other,     D)), "step 28, foil (envelope only)")

import json as _json, os
bat = os.path.join(ROOT, "verification", "battery.json")
if os.path.exists(bat):
    b = _json.load(open(bat))
    steps = [s for s in cfg.steps_ms]
    fig, ax = plt.subplots(figsize=(8.5, 3.8))
    for lv, c, mk in (("rising", "#1f77b4", "o-"), ("ungrouped", "#2ca02c", "s--")):
        y = [b["observers"]["envelope only"].get(f"{lv}:{s:g}", {}).get("cv_dprime", np.nan) for s in steps]
        ax.plot(steps, y, mk, color=c, label=f"'{lv}' ladder")
    ax.axhspan(-0.5, 0.5, color="0.88", alpha=0.9, zorder=0)
    ax.axhline(0, color="0.4", lw=1)
    ax.set(xlabel="onset step between components (ms)",
           ylabel="d' of an envelope-only observer",
           title="Where the non-binding route lives (from the repository's full battery)")
    ax.legend(fontsize=8.5); plt.tight_layout(); plt.show()
    print("The cue is confined to steps 0 and 5 ms and is gone from 10 ms on.")
    print("`seqsfg analyze` reads this profile and marks exactly those cells in its own output,")
    print("so the caveat travels with the data instead of living in a README.")

### 7b. A recurring channel is a rhythm, and that cannot be removed either

A channel that recurs six times at 4 Hz carries six quasi-periodic onsets; in the foil no channel
does. That *is* recurrence, seen one channel at a time, and no construction that keeps "new pitches
every time" can remove it.

At 1.5 Hz it sat inside the noise. At 3–5 Hz it became a real cue — the battery caught it, and a sweep
of ten configurations traded it back down by using six elements instead of eight and a denser cloud.
What survives is bounded: as a fixed rule it reaches d′ ≈ 0.34 against a relabelling ceiling of 0.37,
it does **not** vary with `step`, so it can only add a constant floor to the psychometric function —
it cannot shape it, and a threshold read off the fall-off is unaffected by a constant.

The `onechannel` control cell measures it in a real listener, where it is the *only* cue available.

In [ ]:
#@title Hear the residual: one channel recurring, with no group at all
tone = make_trial(cfg, seed=808, step_ms=0.0, variant="onechannel", d=D)
play(render_interval(cfg, tone.recurring, D),
     "The 'onechannel' control: ONE channel returns at the element times, nothing groups")
play(render_interval(cfg, tone.other, D), "Its foil: a plain cloud")
display(Markdown("If a listener scores above chance on that pair, they are using single-channel "
                 "rhythm rather than binding — which is exactly what the control is for."))

---

## 8. *"Your analysis will find a threshold in noise. They always do."*

Then let's feed it noise and watch.

Below, a simulated listener who is **exactly at chance at every step** — pure coin flips — is pushed
through the real analysis. A logistic function fitted to that data will happily return a threshold.
The question is whether the code *reports* it.

Six gates have to pass before a threshold is printed: the easiest condition must beat chance,
performance must peak at the easiest condition, the threshold must fall inside the tested range, the
bootstrap interval must be narrower than that range, the bootstrap must be defined in ≥ 75% of
resamples, the fit must converge, and the fitted transition must be resolvable by the step spacing.

In [ ]:
#@title Feed the analysis pure noise and watch it refuse
import math
rng = np.random.default_rng(0)

def fake(pc_fn, variant="rising", n_per=40):
    out = []
    for i, s in enumerate(list(cfg.steps_ms) * n_per):
        tgt = 1 + (i % 2); c = int(rng.random() < pc_fn(s))
        out.append(dict(block="main", variant=variant, step_ms=float(s), target_position=tgt,
                        response=tgt if c else 3 - tgt, correct=c))
    return out

for label, fn in (("a listener at chance everywhere", lambda s: 0.5),
                  ("a listener who is perfect everywhere", lambda s: 0.99),
                  ("a real psychometric function (tau = 12 ms)", lambda s: 0.5 + 0.48 * math.exp(-s / 12.0))):
    tr_ = fake(fn)
    ps  = analysis.psychometric_report(analysis.summarize(tr_), cfg.replace(bootstrap_n=200), rng)
    print(f"\n=== {label} ===")
    for g, ok in ps["gates"].items():
        print(f"   [{'ok  ' if ok else 'FAIL'}] {g}")
    if ps["reportable"]:
        print(f"   --> threshold {ps['threshold']:.1f} ms  CI [{ps['threshold_ci'][0]:.1f}, {ps['threshold_ci'][1]:.1f}]")
    else:
        print(f"   --> WITHHELD. (The raw fit would have said {ps['threshold']:.1f} ms. It is noise.)")

---

## 9. Playground — build your own stimulus

Everything above used the shipped configuration. Change it. The validator will refuse combinations
that are physically impossible and **tell you what to change** rather than silently producing a
broken stimulus — try `TONES_PER_CHANNEL = 70` (occupancy too high to pack) or `POOL_LOW_HZ = 100`
(adjacent channels beat slowly enough to be heard as a throb).

In [ ]:
#@title Build a stimulus from scratch   { run: "auto" }
STEP_MS           = 10     #@param {type:"slider", min:0, max:28, step:1}
N_COMPONENTS      = 7      #@param {type:"slider", min:2, max:10, step:1}
TONE_DUR_MS       = 30     #@param {type:"slider", min:15, max:60, step:5}
TONES_PER_CHANNEL = 31     #@param {type:"slider", min:12, max:70, step:1}
POOL_LOW_HZ       = 250    #@param {type:"slider", min:100, max:500, step:25}
VARIANT           = "rising"  #@param ["rising", "ungrouped", "scrambled", "redrawn", "onechannel"]

try:
    my = cfg.replace(n_components=N_COMPONENTS, tone_dur_ms=float(TONE_DUR_MS),
                     tones_per_channel=TONES_PER_CHANNEL, pool_low_hz=float(POOL_LOW_HZ),
                     steps_ms=tuple(sorted({0.0, float(STEP_MS)})),
                     control_cells=(("onechannel", 0.0),),
                     practice_cells=(("rising", 0.0),),
                     main_variants=("rising",))
    myD = config.validate(my)
except config.ConfigError as e:
    print("The validator refused this combination:\n")
    print(e)
else:
    t_ = make_trial(my, seed=1234, step_ms=float(STEP_MS), variant=VARIANT, d=myD)
    xa_, xb_ = render_interval(my, t_.recurring, myD), render_interval(my, t_.other, myD)
    print(f"{myD.n_channels} channels, {myD.channel_freqs_hz[0]:.0f}-{myD.channel_freqs_hz[-1]:.0f} Hz | "
          f"slowest adjacent beat {np.diff(myD.channel_freqs_hz).min():.0f} Hz | "
          f"occupancy {myD.occupancy_per_channel:.2f} | {myD.mean_simultaneous:.1f} tones sounding | "
          f"element span {(N_COMPONENTS - 1) * STEP_MS + TONE_DUR_MS:.0f} ms")
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6), sharey=True)
    for ax_, iv_, ti in ((axes[0], t_.recurring, "target: figure recurs"),
                         (axes[1], t_.other, "foil")):
        plots._raster(ax_, my, myD, iv_, 0.0, my.interval_dur_ms / 1000.0, lw=2.2, boxes=True)
        ax_.set_title(ti); ax_.set_xlabel("time (s)")
        for s_ in ("top", "right"):
            ax_.spines[s_].set_visible(False)
    axes[0].set_ylabel("semitones re 1 kHz")
    plt.tight_layout(); plt.show()
    play(xa_, "target interval"); play(xb_, "foil interval")
    play(render_trial(my, t_, 1, myD), "the whole trial (target first)")

---

## 10. Where this leaves us

**What the verification establishes.** On the `rising` ladder — the one that carries the inference —
every ideal observer restricted to a single property sits at chance after correction, 0 of 30
observer × condition cells survive multiplicity correction, and the largest effect across 62 features
is smaller than relabelling produces. The two intervals are matched in total tones, instantaneous
density, long-term RMS, per-channel spectrum, occupancy, element-locked envelope and element-to-element
loudness. What distinguishes the target is the *conjunction* of channels and relative timing, which is
what binding means.

**What it does not establish.** Three things, all in the README's "what is not controlled" section:

1. **Whether a budget-matched figure is audible at all.** In the published stimulus the figure's
   channels gain energy; here they do not, by design. No battery can answer this — only a pilot can,
   which is what the practice criterion is for. If listeners cannot pass practice at step 0, the design
   has no dynamic range and the budget rule is too strict for this density.
2. **Shearing in time is two manipulations at once.** Delaying the components also makes the element
   longer (30 → 198 ms) and reduces how many sound at once (7 → 2). "Stopped binding" and "became a
   longer, slower object" are the same manipulation here. The control that separates them is a 2 × 2
   crossing onset arrangement with component duration — proposed in the README, not implemented.
3. **The ladder stops short of full non-overlap.** At 5 Hz the widest element that fits is 198 ms, so
   the largest step is 28 ms and adjacent components still overlap by 7%.

**If you want to attack it further**, the highest-value targets are: run the self-test in section 2
at every step and see whether *your own* curve falls where the design predicts; re-run section 5 with
a larger `N_PER_CELL` and a fresh seed; and change something in section 9 that you think should break
the design, then check whether the battery notices.

---

### Where everything lives

| | |
|---|---|
| repository | https://github.com/MeysamAmirsardari/SeqSFG_task |
| full verification report | `verification/battery_report.txt` |
| the 13 diagnostic figures | `verification/figures/` |
| what is *not* controlled | `README.md`, section 4 |
| what a reviewer will push on | `README.md`, section 4b |

```bash
seqsfg config                    # the validated configuration and the ladder
seqsfg verify --trials 40        # the full battery and the ideal observers
seqsfg plots                     # the diagnostic figures
seqsfg demo --step 10 --split    # write one trial to WAV
seqsfg run --data data           # run a real session
seqsfg analyze data/P01/session_01
```

In [ ]:
#@title The verification report, in full (the thing a reviewer reads)
import os
p = os.path.join(ROOT, "verification", "battery_report.txt")
print(open(p).read() if os.path.exists(p) else "run `seqsfg verify --trials 40` to generate this")